In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [2]:
from catboost import CatBoostRegressor 
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score


import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [3]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID','efs_time'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])
print(train.isnull().sum())

dri_score                   154
psych_disturb              2062
cyto_score                 8068
diabetes                   2119
hla_match_c_high           4620
hla_high_res_8             5829
tbi_status                    0
arrhythmia                 2202
hla_low_res_6              3270
graft_type                    0
vent_hist                   259
renal_issue                1915
pulm_severe                2135
prim_disease_hct              0
hla_high_res_6             5284
cmv_status                  634
hla_high_res_10            7163
hla_match_dqb1_high        5199
tce_imm_match             11133
hla_nmdp_6                 4197
hla_match_c_low            2800
rituximab                  2148
hla_match_drb1_low         2643
hla_match_dqb1_low         4194
prod_type                     0
cyto_score_detail         11923
conditioning_intensity     4789
ethnicity                   587
year_hct                      0
obesity                    1760
mrd_hct                   16597
in_vivo_

In [4]:
encoder = LabelEncoder()
normalize = MinMaxScaler()

In [5]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]]=normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        train[i[0]].fillna(train[i[0]].mean(), inplace=True)
        train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]]=normalize.fit_transform(numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))).reshape(-1,1))
    else:
        test[i[0]].fillna(test[i[0]].mean(), inplace=True)
        test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [6]:
train = train.fillna(0)
test = test.fillna(0)

train_y = train['efs']
train_x = train.drop(columns=['efs'])

for i in test:
    test[i]=test[i].to_numpy()
train.head()

,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,...,hepatic_mild,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10,efs
0,0.636364,0.0,1.000000,0.0,0.882258,0.8128,0.000000,0.0,1.0,0.0,...,0.000000,1.00,0.666667,0.5,1.0,0.0,1.0,0.000000,1.0,0.0
1,0.181818,0.0,0.142857,0.0,1.000000,1.0000,0.857143,0.0,1.0,1.0,...,0.000000,0.75,0.333333,0.5,1.0,0.0,1.0,0.666667,1.0,1.0
2,0.636364,0.0,1.000000,0.0,1.000000,1.0000,0.000000,0.0,1.0,0.0,...,0.000000,0.75,0.333333,0.5,1.0,0.0,1.0,0.000000,1.0,0.0
3,0.000000,0.0,0.142857,0.0,1.000000,1.0000,0.000000,0.0,1.0,0.0,...,0.666667,0.75,0.666667,0.5,1.0,0.0,1.0,0.000000,1.0,0.0
4,0.000000,0.0,1.000000,0.0,1.000000,1.0000,0.000000,0.0,1.0,1.0,...,0.000000,0.75,0.333333,0.0,1.0,0.0,1.0,0.000000,1.0,0.0


In [7]:
XGB_hyperparams = {'colsample_bytree': 0.6679438268550072,
                   'gamma': 2,
                   'learning_rate': 0.19750405984350627,
                   'max_depth': 20,
                   'min_child_weight': 9,
                   'n_estimators': 2338,
                   'objective': 'binary:logistic',
                   'reg_alpha': 44,
                   'reg_lambda': 0.5688906203980144,
                   'subsample': 0.5716652066840949}

In [8]:
FOLDS = 5
kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)
    
oof_xgb = numpy.zeros(len(train))
pred_xgb = numpy.zeros(len(test))

for i, (train_index, test_index) in enumerate(kf.split(train)):

    print("#"*25)
    print(f"### Fold {i+1}")
    print("#"*25)
    
    x_test = test.copy()
    
    x_train, x_valid, y_train, y_valid = train_test_split(train.drop(columns=['efs']), train['efs'], 
                                                          test_size = 0.2, 
                                                          random_state = 0)
    model_xgb = XGBRegressor(
        device="cuda",
        max_depth=7,  
        colsample_bytree=0.6, 
        subsample=1, 
        n_estimators=100000,  
        learning_rate=0.09, 
        eval_metric="mae",
        early_stopping_rounds=25,
        objective='reg:logistic',
        enable_categorical=True,
        min_child_weight=5
    )
    
    model_xgb.fit(
        x_train, y_train,
        eval_set=[(x_valid, y_valid)],  
        verbose=100 
    )

    # INFER OOF
    oof_xgb[test_index] = model_xgb.predict(x_valid)
    # INFER TEST
    pred_xgb += model_xgb.predict(x_test)

# COMPUTE AVERAGE TEST PREDS
pred_xgb /= FOLDS
test_predictions=pred_xgb

#########################
### Fold 1
#########################
[0]	validation_0-mae:0.49108
[100]	validation_0-mae:0.40250
[200]	validation_0-mae:0.39485
[300]	validation_0-mae:0.39065
[400]	validation_0-mae:0.38802
[500]	validation_0-mae:0.38549
[600]	validation_0-mae:0.38310
[683]	validation_0-mae:0.38220
#########################
### Fold 2
#########################
[0]	validation_0-mae:0.49108
[100]	validation_0-mae:0.40250
[200]	validation_0-mae:0.39485
[300]	validation_0-mae:0.39065
[400]	validation_0-mae:0.38802
[500]	validation_0-mae:0.38549
[600]	validation_0-mae:0.38310
[682]	validation_0-mae:0.38223
#########################
### Fold 3
#########################
[0]	validation_0-mae:0.49108
[100]	validation_0-mae:0.40250
[200]	validation_0-mae:0.39485
[300]	validation_0-mae:0.39065
[400]	validation_0-mae:0.38802
[500]	validation_0-mae:0.38549
[600]	validation_0-mae:0.38310
[683]	validation_0-mae:0.38220
#########################
### Fold 4
#########################
[0]	valida

lgbm_hyperparams={ 'colsample_bytree': 0.3394676841949491,
                   'drop_rate': 0.005229247069083676,
                   'learning_rate': 0.08939376710901481,
                   'max_bin': 3192,
                   'max_depth': 5655,
                   'max_drop': 3654,
                   'min_child_samples': 9011,
                   'min_data_in_leaf': 1320,
                   'n_estimators': 5619,
                   'num_leaves': 8002,
                   'objective': 'regression_l1',
                   'reg_alpha': 0.3092258349709437,
                   'reg_lambda': 0.5130123398875309,
                   'skip_drop': 0.5672900003846416,
                   'verbosity': -1}

XGB_hyperparams = {'colsample_bytree': 0.6679438268550072,
                   'gamma': 2,
                   'learning_rate': 0.19750405984350627,
                   'max_depth': 20,
                   'min_child_weight': 9,
                   'n_estimators': 2338,
                   'objective': 'binary:logistic',
                   'reg_alpha': 44,
                   'reg_lambda': 0.5688906203980144,
                   'subsample': 0.5716652066840949}



lightgbm = AdaBoostRegressor(LGBMRegressor(**lgbm_hyperparams),
                             random_state=42, 
                             n_estimators=4,
                             learning_rate=0.1)

xgboost = AdaBoostRegressor(XGBRegressor(n_estimators=int(XGB_hyperparams['n_estimators']),  
                                         learning_rate=XGB_hyperparams['learning_rate'],
                                         colsample_bytree=XGB_hyperparams['colsample_bytree'], 
                                         subsample=XGB_hyperparams['subsample'], 
                                         objective='binary:logistic',
                                         min_child_weight=XGB_hyperparams['min_child_weight'],
                                         gamma=XGB_hyperparams['gamma'],
                                         max_depth=int(XGB_hyperparams['max_depth']),
                                         reg_alpha=XGB_hyperparams['reg_alpha'],
                                         reg_lambda=XGB_hyperparams['reg_lambda']),
                            random_state=42, 
                            n_estimators=4,
                            learning_rate=0.1)

#reg:logistic
#binary:logitraw
#binary:logistic

model = StackingRegressor(estimators=[('xgboost', XGBRegressor()), 
                                      ('lightgbm', LGBMRegressor()), 
                                      ('ridgecv', RidgeCV()),
                                      ('lasso', LassoCV()), 
                                      ('elasticnet', ElasticNetCV()), 
                                      ('gbr', GradientBoostingRegressor())])


model.fit(X_train, y_train)

In [9]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']

test_predictions = model.predict(test)

mean_absolute_error(y_test, model.predict(X_test))

mean_absolute_error(y_test, model_xgb.predict(X_test))

In [10]:
submission = pandas.DataFrame({
    'ID': id.values,
    'prediction': test_predictions,
})
submission

,ID,prediction
0,28800,0.001223
1,28801,0.905056
2,28802,0.000526


In [11]:
submission.to_csv('submission.csv', index = False)